# Basic

In [1]:
%load_ext autoreload
%autoreload all

In [2]:
import polars as pl
import pickle
import numpy as np
import tqdm
import os
import glob
import multiprocessing as mp
import networkx as nx

import src.graph_tokenizer_gd_tree_dev.config as config
import src.graph_tokenizer_gd_tree_dev.tokenizer as tokenizer
import src.graph_tokenizer_gd_tree_dev.eval as eval
import src.graph_tokenizer_gd_tree_dev.graph_fct as graph_fct
import src.graph_tokenizer_gd_tree_dev.utils as utils

In [3]:
Ks = config.TokenizerParam().Ks
rnd_iters = config.TokenizerParam().rnd_iters

In [4]:
id_to_label, combined_subgraphs = graph_fct.get_combined_combined_subgraphs_and_id2label()
df_mapped = pl.read_parquet(f"{config.BasicConfig().mapped_path}")
mapped_ids = df_mapped["id"].unique().to_list()

gd_tree_list = glob.glob(config.CandidateLists().path_greedy_tree + '/*.parquet')
baseline_list = glob.glob(config.CandidateLists().baseline_path + '/*.parquet')

D = config.TokenizerParam().max_dist_candidate
D

9

In [5]:
all_candidates = {
    "greedy_tree_margin": utils.to_type_dict(gd_tree_list),
    "baseline": utils.to_type_dict(baseline_list),
}
for category, file_list in all_candidates.items():
    for file_type, file in file_list.items():
        print(f"Category: {category}, File Type: {file_type}, File: {file}")

Category: greedy_tree_margin, File Type: 0.0, File: D:/greedy_graph_data/greedy_tree_candidates\0.0.parquet
Category: greedy_tree_margin, File Type: 0.2, File: D:/greedy_graph_data/greedy_tree_candidates\0.2.parquet
Category: greedy_tree_margin, File Type: 0.4, File: D:/greedy_graph_data/greedy_tree_candidates\0.4.parquet
Category: greedy_tree_margin, File Type: 0.6, File: D:/greedy_graph_data/greedy_tree_candidates\0.6.parquet
Category: greedy_tree_margin, File Type: 0.8, File: D:/greedy_graph_data/greedy_tree_candidates\0.8.parquet
Category: greedy_tree_margin, File Type: 1.0, File: D:/greedy_graph_data/greedy_tree_candidates\1.0.parquet
Category: baseline, File Type: closeness_centrality, File: D:/greedy_graph_data/baseline_candidates\closeness_centrality.parquet
Category: baseline, File Type: discrete_set_cover, File: D:/greedy_graph_data/baseline_candidates\discrete_set_cover.parquet
Category: baseline, File Type: highest_degree, File: D:/greedy_graph_data/baseline_candidates\high

# baselines except k rnd

In [ ]:
results = []
candidate_col = "token"
tasks = []
for category, file_list in all_candidates.items():
    if category == "greedy_tree_margin":
        continue
    for file_type, file in file_list.items():
        if file_type == "k_random_all_samples":
            continue

        df = pl.read_parquet(file)
        for k in Ks:
            candidates = df.head(k)[candidate_col].to_list()
            tasks.append(((category, file_type, k), candidates))

# Built once and shared across every task instead of being rebuilt per (category, file_type, k):
# A/node_to_idx is the semantic-coverage transition matrix, adj is the out-adjacency used by
# context-tree building. Neither depends on the candidate set T, only on the fixed graph.
A, node_to_idx = tokenizer.build_coverage_transition(combined_subgraphs)
adj = tokenizer.build_out_adjacency(combined_subgraphs)

n_workers = os.cpu_count()
with mp.Pool(n_workers, initializer=tokenizer._init_coverage_worker,
             initargs=(A, node_to_idx, adj, mapped_ids, D, id_to_label)) as pool:
    task_results = pool.imap_unordered(tokenizer._worker_coverage_score, tasks, chunksize=4)
    for (category, file_type, k), metrics in tqdm.tqdm(task_results, total=len(tasks)):
        results.append({
            "category": category,
            "file_type": file_type,
            "k": k,
            **metrics,
        })

results_df = pl.DataFrame(results)
results_df.write_parquet(config.Results().perf_baseline_path)

# random K

In [ ]:
results_k = []
candidate_col = "token"
category, file_type = "baseline", "k_random_all_samples"
df = pl.read_parquet(config.CandidateLists().k_random_all_samples)
tasks = [
    ((k, it), df.filter((pl.col("iter") == it) & (pl.col("k") == k))[candidate_col].to_list())
    for k in Ks
    for it in rnd_iters
]
n_workers = os.cpu_count()
with mp.Pool(n_workers, initializer=tokenizer._init_coverage_worker,
             initargs=(A, node_to_idx, adj, mapped_ids, D, id_to_label)) as pool:
    task_results = pool.imap_unordered(tokenizer._worker_coverage_score, tasks, chunksize=4)
    for (k, it), metrics in tqdm.tqdm(task_results, total=len(tasks)):
        results_k.append({
            "category": category,
            "file_type": file_type,
            "k": k,
            "iter": it,
            **metrics,
        })

results_k_df = pl.DataFrame(results_k)
results_k_df.write_parquet(config.Results().perf_k_rdn_path)


# greedy

In [6]:
results = []
candidate_col = "token"
tasks = []
for category, file_list in all_candidates.items():
    if category != "greedy_tree_margin":
        continue
    for file_type, file in file_list.items():
        df = pl.read_parquet(file)
        for k in Ks:
            candidates = df.head(k)[candidate_col].to_list()
            tasks.append(((category, file_type, k), candidates))

# Built once and shared across every task instead of being rebuilt per (category, file_type, k):
# A/node_to_idx is the semantic-coverage transition matrix, adj is the out-adjacency used by
# context-tree building. Neither depends on the candidate set T, only on the fixed graph.
A, node_to_idx = tokenizer.build_coverage_transition(combined_subgraphs)
adj = tokenizer.build_out_adjacency(combined_subgraphs)
len(tasks)

234

In [7]:

n_workers = os.cpu_count()
with mp.Pool(n_workers, initializer=tokenizer._init_coverage_worker,
             initargs=(A, node_to_idx, adj, mapped_ids, D, id_to_label)) as pool:
    task_results = pool.imap_unordered(tokenizer._worker_coverage_score, tasks, chunksize=4)
    for (category, file_type, k), metrics in tqdm.tqdm(task_results, total=len(tasks)):
        results.append({
            "category": category,
            "file_type": file_type,
            "k": k,
            **metrics,
        })

results_df = pl.DataFrame(results)
results_df.write_parquet(config.Results().perf_greedy_tree_path)

100%|██████████| 234/234 [1:06:10<00:00, 16.97s/it]
